## Changelog

| Version | Date | Changes |
|---------|------|---------|
| v1 | 2026-07-20 | Initial bump chart - all visible topics, plain circle nodes |
| v2 | 2026-07-20 | Limit chart to top 30 topics; node labels use bold text with bounding box |
| v3 | 2026-07-20 | Vertical per-hour gridlines; dual top+bottom x-axis with UTC/GMT+8/GMT+7 labels; `TOP_N` configurable from config cell |
| v4 | 2026-07-20 | Nodes are now boxed keyword labels at every hour position, replacing dot markers |
| v5 | 2026-07-20 | Fill all rank slots 1..TOP_N: replaced head(top_n) topic cap with min(rank) <= top_n filter so every position is always covered |
| v6 | 2026-07-20 | Readable static output (2x font sizes, first+last-only labels per topic); dark-mode PNG variant (`_dark.png`); Plotly interactive HTML with hover-highlight and topic search box (`_interactive.html`); `build_rank_pivot` shared helper extracted from renderer; `refresh_latest_pointers` stable latest-copy outputs for future GitHub Pages |
| v7 | 2026-07-20 | Interactive chart overhaul: boxed first+last labels matching static chart; scrollable x-axis at natural pixel width; unified JS state model fixing search+hover conflict; per-topic checkbox list for hard show/hide; client-side time-range dropdown (Last 6h/24h/3d/7d/All) driven by embedded combined-dataset JSON; `build_combined_csv` rebuilds on every run with latest-wins dedupe on `(name_norm, hour_label)`; Alte Haas Grotesk font via CDN fallback; light/dark mode toggle with localStorage; dual top+bottom x-axes via Plotly xaxis2; sticky y-axis panel; ARIA labels + keyboard navigation; click-to-pin trace emphasis; reset view button; download CSV for current range; URL query-param sync; debounced search (150 ms); topic stats readout in floating panel; collapsible sidebar; `OUTPUT_ROOT` anchored to repo root via `.git` detection; `get_first_last_positions` shared helper extracted from `render_bump_chart` |
| v8 | 2026-07-20 | Bug fixes: (1) inject Plotly.js via plotly.offline.get_plotlyjs() instead of regex-extracting first script from to_html() output; (2) fix _ts() timestamp parser so hour-column timestamps are real Unix epochs, making the time-range dropdown actually filter |
| v9 | 2026-07-20 | Fixed IIFE scope bug: removed inline `on*` event attributes from toolbar and stats-panel; wired all controls with `addEventListener` inside the closure. Added **Select All / Deselect All** buttons to the topic sidebar. |
| v10 | 2026-07-20 | Four interactive chart fixes: (1) Replaced toolbar chart-dimming search with sidebar row-filter search (`#ssi`) above Topics header — filters checkbox rows by substring, never touches chart opacity, removes `S.search`/`schS`/`clrS`/`filter` URL param; (2) `rebuildCb(hi)` now accepts filtered hour-index array and only renders rows for topics present in the current time range — sidebar shrinks/grows with range dropdown; (3) Day-boundary markers: thick dashed vertical Plotly shapes + stacked tri-timezone annotations (UTC, GMT+8, GMT+7) in format "{DayName}, DD MM YYYY ({tz})", recomputed on every range change; (4) Dynamic chart width: removed Python-side `w`/`__CW__` bake-in — `buildLayout(hi)` computes `Math.max(containerW, hi.length*80)` at redraw time so short ranges fill the pane and horizontal scroll only triggers when needed. |
| v11 | 2026-07-21 | Date-range indicator in toolbar: compact `<span id="drng">` inserted between Download CSV and the flex spacer, populated by `updDrng(hi)` called from `redraw()`. Shows start/end of currently displayed range as `[DayName, DD Mon YYYY, UTC: HH:MM, GMT+8: HH:MM, GMT+7: HH:MM] to [...]`. Shared module-level `DAYS`/`MONS`/`tzList` constants and `fmtTs(epochSec,off)` helper replace inline date math in `buildDayBoundaries`. |
| v12 | 2026-07-21 | Changed default toolbar time range from All to Last 6h: moved `selected` attribute in the `#rng` option list from the All option to the 6h option, and changed JS state default `S.range` from "All" to "6h"; updated `syncUrl()` so the URL `range` param is omitted only when `S.range==="6h"` (the new default), keeping links like `?range=7d`/`?range=All` working via the existing `p.has("range")` override. |
| v13 | 2026-07-21 | Fixed bug where unchecking a topic in the sidebar hid its line but left its boxed keyword-label annotations on screen, overlapping labels of topics still selected: `buildAnns(hi)` now skips topics with `S.checked[t.n]===false`; `redraw()` stores `S.hi`/`S.dbAnns` for reuse outside the render cycle; `applyVis()` now calls `Plotly.relayout(gd,{annotations:...})` after its opacity `Plotly.restyle()` so annotations stay in sync with checkbox state on every toggle, hover, pin, and range change. |

---

## Part A - Framework

Imports, logging configuration, and all helper functions.

Functions are written as plain importable callables - not notebook-only code - so they
can be moved to `scripts/capture_trends.py` for the GitHub Actions cron layer without
modification (local-first-now / automation-later seam).

**robots.txt (trends24.in):** The site blocks named AI-training crawlers (ClaudeBot,
GPTBot, Google-Extended, etc.) but permits the generic `User-agent: *` catch-all
(`Disallow:` empty = allow all). `Content-Signal: ai-train=no, use=reference` - reading
ranked names into a chart is reference use, not model training. A plain `requests`
script with an honest, non-spoofed User-Agent is permitted.

**Selectors note:** `fetch_page` + `parse_timeline_columns` use `.list-container` /
`h3` / `ol li a` selectors that match the `context.md` sketch and align with
HamidByte's public scraper targeting the same site structure. `parse_detail_stats_table`
uses selectors from the same scraper's worldwide-page implementation; they have *not*
been confirmed against `/indonesia/` specifically - treat as a strong hypothesis.


In [1]:
import json
import re
import hashlib
import logging
import shutil
from datetime import datetime, timezone
from pathlib import Path
import colorsys

import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import ticker
import plotly.graph_objects as go

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("kestrel-trends")

In [3]:
def normalize_trend_name(name: str) -> str:
    """Strip leading '#', collapse whitespace, lowercase - cross-hour/cross-source dedup key."""
    return re.sub(r"\s+", " ", name.lstrip("#")).strip().lower()


def parse_tweet_count(raw: str) -> int | None:
    """Parse human-readable count like '74M Tweets' â†’ 74_000_000; '1.2K' â†’ 1200.
    Returns None on empty input or unrecognised format."""
    if not raw:
        return None
    m = re.match(r"([\d.]+)\s*([KMBkmb]?)", raw.replace(",", ""))
    if not m:
        return None
    n, suffix = float(m.group(1)), m.group(2).upper()
    mult = {"": 1, "K": 1_000, "M": 1_000_000, "B": 1_000_000_000}.get(suffix)
    return int(n * mult) if mult is not None else None


def parse_duration(raw: str) -> tuple[str, float | None]:
    """Parse trending-duration string like '2 hours 30 minutes' â†’ (label, 2.5).

    Returns (raw_label, total_hours). total_hours is None if no time pattern matched.
    Handles: days, hours/hr, minutes/min, seconds/sec (case-insensitive, plural-tolerant).
    """
    label = (raw or "").strip()
    if not label:
        return (label, None)
    total_hours = 0.0
    found = False
    for qty_str, unit in re.findall(
        r"(\d+(?:\.\d+)?)\s*(day|hour|hr|minute|min|second|sec)s?", label, re.I
    ):
        qty = float(qty_str)
        u = unit.lower()
        if u.startswith("day"):
            total_hours += qty * 24
        elif u.startswith("hour") or u == "hr":
            total_hours += qty
        elif u.startswith("min"):
            total_hours += qty / 60
        elif u.startswith("sec"):
            total_hours += qty / 3600
        found = True
    return (label, total_hours if found else None)


def shorten_hour_label(label: str) -> str:
    """Extract HH:MMZ from a JS Date toString string for chart axis display.

    Trends24 h3 text is a full JS Date string:
      'Sun Jul 19 2026 16:40:28 GMT+0000 (Coordinated Universal Time)' â†’ '16:40Z'
    Falls back to first 12 chars if the pattern doesn't match.
    The raw label is preserved in the CSV; this is chart-display-only formatting.
    """
    m = re.search(r"(\d{1,2}:\d{2}):\d{2}\s*GMT", str(label))
    return (m.group(1) + "Z") if m else str(label)[:12]

def format_tz_label(raw_label: str) -> str:
    """Multi-line x-axis label showing UTC / GMT+8 / GMT+7 for a raw JS Date string.

    Extracts HH:MM UTC from the JS Date string, then computes +7 and +8 offsets.
    Returns a 3-line string, e.g.:
      16:40Z
      00:40+8
      23:40+7
    Falls back to shorten_hour_label output if the pattern does not match.
    """
    import re as _re
    m = _re.search(r"(\d{1,2}):(\d{2}):\d{2}\s*GMT", str(raw_label))
    if not m:
        return shorten_hour_label(raw_label)
    total_min = int(m.group(1)) * 60 + int(m.group(2))

    def _fmt(mins: int) -> str:
        return f"{(mins // 60) % 24:02d}:{mins % 60:02d}"

    return f"{_fmt(total_min)}Z\n{_fmt(total_min + 480)}+8\n{_fmt(total_min + 420)}+7"


In [4]:
def fetch_page(url: str, ua: str, timeout: int) -> BeautifulSoup:
    """GET url with honest User-Agent; raise on HTTP error; return parsed BeautifulSoup."""
    resp = requests.get(url, headers={"User-Agent": ua}, timeout=timeout)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def parse_timeline_columns(soup: BeautifulSoup, min_trends: int = 30) -> list[dict]:
    """Parse every visible hour-column from trends24.in/indonesia/.

    Selectors (from context.md sketch + HamidByte scraper targeting trends24.in):
      column wrapper : .list-container
      hour label     : h3 (first h3 inside .list-container)
      trend items    : ol li a  (text = display name; title attr = tweet-count label)

    Grabs ALL visible columns in one request - free backfill redundancy (see context.md Â§4):
    a single missed hourly cron run is recovered by the next successful run as long as the
    site's own retention window covers the gap.

    Raises RuntimeError on schema drift (no columns found, or largest column has fewer
    than min_trends entries). Fail loud - do not silently write partial data.

    Returns list of column dicts:
      {hour_label: str|None,
       trends: [{rank, name, name_norm, tweet_count_raw, tweet_count}]}
    """
    columns = []
    for col_el in soup.select(".list-container"):
        label_el = col_el.select_one("h3")
        hour_label = label_el.get_text(strip=True) if label_el else None
        trends = []
        for rank, a_tag in enumerate(col_el.select("ol li a"), start=1):
            name = a_tag.get_text(strip=True)
            if not name:
                continue
            count_raw = a_tag.get("title", "") or ""
            trends.append({
                "rank": rank,
                "name": name,
                "name_norm": normalize_trend_name(name),
                "tweet_count_raw": count_raw if count_raw else None,
                "tweet_count": parse_tweet_count(count_raw),
            })
        if trends:
            columns.append({"hour_label": hour_label, "trends": trends})

    if not columns:
        raise RuntimeError(
            "Schema drift on trends24.in/indonesia/: no '.list-container' columns found. "
            "Open the page in browser devtools and verify the column-wrapper selector - "
            "the site may have changed its markup."
        )
    max_len = max(len(c["trends"]) for c in columns)
    if max_len < min_trends:
        raise RuntimeError(
            f"Schema drift: largest column has only {max_len} trends, "
            f"expected >= {min_trends}. Verify selectors in devtools."
        )
    return columns

In [5]:
def _safe_int(s) -> int | None:
    """Convert string to int, returning None on any type or value error."""
    try:
        return int(str(s).strip())
    except (TypeError, ValueError):
        return None


def parse_detail_stats_table(soup: BeautifulSoup) -> list[dict] | None:
    """Attempt to parse the Table-view detail stats from the same trends24.in page.

    Hypothesis from HamidByte/Twitter-Trending-Hashtags-Scraper-Python (worldwide page):
    the table is pre-rendered in the raw HTML under a CSS-toggled tab, not JS-injected -
    so plain requests+BeautifulSoup is sufficient if the markup is present.

    Selectors tried (in order):
      primary  : section#table .table-container-4 table.the-table tbody.list
      fallback : table.the-table tbody

    Expected column shape per <tr>:
      td.rank                   â†’ current rank in the table listing
      td.topic > a.trend-link   â†’ topic display name (and link)
      td.position               â†’ best position ever reached
      td.count[data-count]      â†’ total tweet count (prefer data-count attr, fall back to text)
      td.duration               â†’ how long the topic has been trending (raw text)

    IMPORTANT: These selectors were confirmed on the worldwide trends24.in root page by
    HamidByte's scraper; they have NOT been confirmed against /indonesia/ specifically.
    Treat as a strong starting hypothesis. If this function returns None, check whether:
      (a) the /indonesia/ subpage simply does not have the table tab, or
      (b) the table is genuinely JS-injected â†’ install Playwright and re-fetch with it.

    Returns list of detail dicts, or None if the table section is absent from raw HTML.
    When None, best_position/total_tweets/trending_for_* will be null in the output CSV.
    """
    tbody = soup.select_one(
        "section#table .table-container-4 table.the-table tbody.list"
    )
    if tbody is None:
        tbody = soup.select_one("table.the-table tbody")  # looser fallback
    if tbody is None:
        return None

    rows = []
    for tr in tbody.select("tr"):
        topic_a = tr.select_one("td.topic a.trend-link")
        if not topic_a:
            continue
        rank_el = tr.select_one("td.rank")
        pos_el = tr.select_one("td.position")
        count_el = tr.select_one("td.count")
        dur_el = tr.select_one("td.duration")

        count_raw = ""
        if count_el:
            count_raw = count_el.get("data-count", "") or count_el.get_text(strip=True)

        dur_label, dur_hours = parse_duration(
            dur_el.get_text(strip=True) if dur_el else ""
        )
        rows.append({
            "rank_table": _safe_int(rank_el.get_text(strip=True) if rank_el else None),
            "name": topic_a.get_text(strip=True),
            "name_norm": normalize_trend_name(topic_a.get_text(strip=True)),
            "best_position": _safe_int(pos_el.get_text(strip=True) if pos_el else None),
            "total_tweets": parse_tweet_count(count_raw),
            "trending_for_raw": dur_label if dur_label else None,
            "trending_for_hours": dur_hours,
        })
    return rows if rows else None

In [6]:
def build_pumpkin_palette(n: int = 150) -> list[str]:
    """Build a preset palette of n warm-autumn hex colors anchored on Pumpkin (#FF7518).

    Grid layout: 6 hue × 5 saturation × 5 lightness = 150 total.

    Hue range [4°, 69°]: pumpkin base ≈ 24° (spread -20° / +45°), covering
    deep red-orange → orange → burnt-orange → amber in one cohesive warm family.
    Saturation range [70%, 100%], Lightness range [35%, 65%].

    Colors are generated once (in Part B) and kept stable across runs.
    Topic → color assignment is via get_topic_color(name_norm, palette) which hashes
    name_norm modulo palette length - same topic always gets the same color.

    colorsys note: hls_to_rgb(h, l, s) takes HLS order (not HSL).
    """
    assert n == 150, (
        f"Grid 6×5×5=150 exactly; got n={n}. "
        "Adjust n_h/n_s/n_l and the assert if you need a different size."
    )
    n_h, n_s, n_l = 6, 5, 5
    hue_min, hue_max = 4 / 360, 69 / 360   # [4°, 69°] as colorsys [0,1] fractions
    sat_min, sat_max = 0.70, 1.00
    lit_min, lit_max = 0.35, 0.65

    hues = [hue_min + i * (hue_max - hue_min) / (n_h - 1) for i in range(n_h)]
    sats = [sat_min + i * (sat_max - sat_min) / (n_s - 1) for i in range(n_s)]
    lits = [lit_min + i * (lit_max - lit_min) / (n_l - 1) for i in range(n_l)]

    palette: list[str] = []
    for h in hues:
        for s in sats:
            for l in lits:
                # colorsys.hls_to_rgb takes (hue, lightness, saturation)
                r, g, b = colorsys.hls_to_rgb(h, l, s)
                r_i = min(255, int(r * 255))
                g_i = min(255, int(g * 255))
                b_i = min(255, int(b * 255))
                palette.append(f"#{r_i:02x}{g_i:02x}{b_i:02x}")
    return palette


def build_pumpkin_palette_dark(n: int = 150) -> list[str]:
    """Dark-mode variant: same hue/sat range as build_pumpkin_palette, lightness 55–82%.

    The light palette uses L=[35%, 65%] tuned for white backgrounds; on a dark
    background those values look muddy and low-contrast. Raising the lightness
    range produces the same warm pumpkin family at higher luminosity, which reads
    as vivid against #121212 without shifting the hue character of the palette.

    colorsys note: hls_to_rgb(h, l, s) takes HLS order (not HSL).
    """
    assert n == 150, (
        f"Grid 6×5×5=150 exactly; got n={n}. "
        "Adjust n_h/n_s/n_l and the assert if you need a different size."
    )
    n_h, n_s, n_l = 6, 5, 5
    hue_min, hue_max = 4 / 360, 69 / 360
    sat_min, sat_max = 0.70, 1.00
    lit_min, lit_max = 0.55, 0.82   # brighter range for dark backgrounds

    hues = [hue_min + i * (hue_max - hue_min) / (n_h - 1) for i in range(n_h)]
    sats = [sat_min + i * (sat_max - sat_min) / (n_s - 1) for i in range(n_s)]
    lits = [lit_min + i * (lit_max - lit_min) / (n_l - 1) for i in range(n_l)]

    palette: list[str] = []
    for h in hues:
        for s in sats:
            for l in lits:
                r, g, b = colorsys.hls_to_rgb(h, l, s)
                r_i = min(255, int(r * 255))
                g_i = min(255, int(g * 255))
                b_i = min(255, int(b * 255))
                palette.append(f"#{r_i:02x}{g_i:02x}{b_i:02x}")
    return palette


def get_topic_color(name_norm: str, palette: list[str]) -> str:
    """Return a stable color for name_norm: MD5(name_norm) mod len(palette).

    Same name_norm always maps to the same color regardless of run order or
    how many other topics are present - palette reshuffling across runs is avoided.
    """
    idx = int(hashlib.md5(name_norm.encode()).hexdigest(), 16) % len(palette)
    return palette[idx]

In [7]:
def join_data(
    timeline_columns: list[dict],
    detail_stats: list[dict] | None,
    captured_at_utc: datetime,
    run_id: str,
    source_url: str,
) -> pd.DataFrame:
    """Flatten timeline columns and left-join detail stats on name_norm.

    Topics in the timeline that have no matching detail-stats row get None for
    best_position / total_tweets / trending_for_* - they are NOT dropped.
    Topics in detail_stats with no matching timeline entry are simply absent
    from the output (right-side-only rows are not included).

    Output schema (one row per topic per hour-column):
      captured_at_utc, run_id, source_url, hour_label, rank,
      name, name_norm, tweet_count_raw, tweet_count,
      best_position, total_tweets, trending_for_raw, trending_for_hours
    """
    detail_by_norm: dict[str, dict] = {}
    for row in (detail_stats or []):
        norm = row.get("name_norm", "")
        if norm:
            detail_by_norm[norm] = row

    rows = []
    for col in timeline_columns:
        for trend in col["trends"]:
            norm = trend["name_norm"]
            d = detail_by_norm.get(norm, {})
            rows.append({
                "captured_at_utc": captured_at_utc.isoformat(),
                "run_id": run_id,
                "source_url": source_url,
                "hour_label": col["hour_label"],
                "rank": trend["rank"],
                "name": trend["name"],
                "name_norm": norm,
                "tweet_count_raw": trend.get("tweet_count_raw"),
                "tweet_count": trend.get("tweet_count"),
                "best_position": d.get("best_position"),
                "total_tweets": d.get("total_tweets"),
                "trending_for_raw": d.get("trending_for_raw"),
                "trending_for_hours": d.get("trending_for_hours"),
            })
    return pd.DataFrame(rows)


def write_raw_csv(df: pd.DataFrame, path: Path) -> None:
    """Write flat denormalized DataFrame to CSV (UTF-8, no index)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8")
    log.info("CSV written: %s (%d rows)", path, len(df))

In [8]:
def build_rank_pivot(df: pd.DataFrame, top_n: int) -> pd.DataFrame:
    """Pivot df into a (name_norm × hour_label) rank matrix, filtered to top_n.

    Shared by both static and interactive renderers so pivot logic stays DRY.
    Columns ordered oldest-left → newest-right (reversed from pivot_table default).
    Only topics that reach rank <= top_n in at least one hour are included.
    Rows ordered by best (lowest) rank across all hours.
    """
    pivot = df.pivot_table(
        index="name_norm", columns="hour_label", values="rank", aggfunc="min"
    )
    pivot = pivot[pivot.columns[::-1]]
    pivot = pivot.loc[pivot.min(axis=1) <= top_n]
    pivot = pivot.loc[pivot.min(axis=1).sort_values().index]
    return pivot

In [9]:
def get_first_last_positions(y_vals) -> set:
    """Return {first_idx, last_idx} of non-NaN/non-None positions in y_vals.

    Used by both render_bump_chart (static) and render_bump_chart_interactive so the
    same declutter rule applies in both outputs: only the first and last appearance of
    each topic gets a boxed keyword label; intermediate hours show the line only.

    Returns an empty set if all values are NaN or None.
    """
    non_nan = [
        xi for xi, yv in enumerate(y_vals)
        if yv is not None and not (isinstance(yv, float) and pd.isna(yv))
    ]
    if not non_nan:
        return set()
    return {non_nan[0], non_nan[-1]}

In [10]:
def render_bump_chart(
    df: pd.DataFrame,
    palette: list[str],
    out_path: Path,
    top_n: int = 30,
    dark: bool = False,
) -> None:
    """Render and save a rank-over-time bump chart (light or dark mode).

    Layout:
      Y-axis : rank, inverted (rank 1 at the top), capped at top_n
      X-axis : hour_label columns (oldest left, newest right)
               Labels on both top and bottom: UTC / GMT+8 / GMT+7
      Lines  : one per unique topic; NaN hours = gaps (not interpolated)
      Labels : boxed keyword label at first AND last non-NaN appearance only
               (intermediate hours show the colored line without a label box —
               standard bump-chart declutter that keeps text legible at 10pt)
               Label positions determined by get_first_last_positions() so the
               rule is consistent with the interactive renderer.
      Color  : stable per topic via get_topic_color; pass PUMPKIN_PALETTE_DARK
               for the dark variant so lightness is tuned for a dark background
      Grid   : horizontal rank gridlines + vertical per-hour-column gridlines

    dark=True selects a #121212 background, #1c1c1c label-box fill, and adjusted
    grid/spine/tick colors suited for a dark monitor. Pass PUMPKIN_PALETTE_DARK
    (not PUMPKIN_PALETTE) as palette when dark=True so colors are bright enough
    to read against the dark background.
    """
    if df.empty:
        log.warning("DataFrame is empty - skipping bump chart render")
        return

    if dark:
        bg, fg     = "#121212", "#e0e0e0"
        grid_x     = "#383838"
        grid_y     = "#2a2a2a"
        box_bg     = "#1c1c1c"
        spine_c    = "#484848"
        overflow_c = "#bbbbbb"
    else:
        bg, fg     = "white", "black"
        grid_x     = "#999999"
        grid_y     = "#cccccc"
        box_bg     = "white"
        spine_c    = "#cccccc"
        overflow_c = "#555555"

    pivot = build_rank_pivot(df, top_n)
    n_topics       = len(pivot)
    n_hours        = len(pivot.columns)
    palette_size   = len(palette)
    overflow_count = max(0, n_topics - palette_size)

    fig_w = max(20, n_hours * 2.8)
    fig_h = max(14, min(top_n * 0.55 + 6, 52))

    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=100)
    fig.patch.set_facecolor(bg)
    ax.set_facecolor(bg)

    x_pos    = list(range(n_hours))
    x_labels = [format_tz_label(lbl) for lbl in pivot.columns]

    for plot_idx, name_norm in enumerate(pivot.index):
        color     = get_topic_color(name_norm, palette)
        linestyle = "--" if plot_idx >= palette_size else "-"
        y_vals    = pivot.loc[name_norm].values

        ax.plot(
            x_pos, y_vals,
            color=color, linestyle=linestyle,
            linewidth=1.2, alpha=0.60,
            marker=None, zorder=1,
        )

        # First and last non-NaN positions only — shared logic with interactive renderer
        label_positions = get_first_last_positions(y_vals)
        if not label_positions:
            continue

        for xi in label_positions:
            yv = y_vals[xi]
            if pd.isna(yv):
                continue
            ax.annotate(
                name_norm,
                xy=(x_pos[xi], yv),
                xytext=(0, 0), textcoords="offset points",
                fontsize=10, va="center", ha="center",
                color=color, clip_on=True,
                fontweight="bold",
                bbox=dict(
                    boxstyle="square,pad=0.25",
                    facecolor=box_bg,
                    edgecolor=color,
                    linewidth=0.8,
                    alpha=0.93,
                ),
                zorder=3,
            )

    ax.invert_yaxis()
    ax.set_ylim(top_n + 0.5, 0.5)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, rotation=90, ha="center", fontsize=8,
                       linespacing=1.3, color=fg)
    ax.tick_params(axis="x", which="major", bottom=True, labelbottom=True,
                   top=False, labeltop=False, colors=fg)
    ax.tick_params(axis="y", colors=fg)

    ax.grid(True, axis="x", which="major", linestyle=":", alpha=0.40, color=grid_x)
    ax.grid(True, axis="y", which="major", linestyle=":", alpha=0.45, color=grid_y)

    for spine in ax.spines.values():
        spine.set_edgecolor(spine_c)

    ax.yaxis.set_major_locator(ticker.MultipleLocator(5))
    ax.yaxis.set_minor_locator(ticker.MultipleLocator(1))
    ax.set_ylabel("Rank (1 = top trend)", fontsize=11, color=fg)
    ax.set_xlabel("Hour (left = oldest, right = most recent)", fontsize=10, color=fg)
    ax.set_title(
        f"Indonesia Twitter/X Trends - Top {top_n} Bump Chart\n"
        f"Run: {out_path.parent.name}  |  {n_topics} topics  |  {n_hours} hour-columns",
        fontsize=13, pad=16, color=fg,
    )

    ax_top = ax.twiny()
    ax_top.set_facecolor(bg)
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(x_pos)
    ax_top.set_xticklabels(x_labels, rotation=90, ha="center", fontsize=8,
                            linespacing=1.3, color=fg)
    ax_top.tick_params(axis="x", which="major", length=4, pad=3, colors=fg)
    for spine in ax_top.spines.values():
        spine.set_edgecolor(spine_c)

    if overflow_count > 0:
        ax.annotate(
            f"⚠ {overflow_count} topic(s) exceed palette ({palette_size} colors); shown dashed",
            xy=(0.01, 0.01), xycoords="axes fraction", fontsize=7.5, color=overflow_c,
        )

    plt.tight_layout(pad=1.5)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close(fig)
    log.info(
        "Bump chart saved: %s (%d topics, %d hour-columns, dark=%s)",
        out_path, n_topics, n_hours, dark,
    )

In [11]:
def render_bump_chart_interactive(
    df_combined: pd.DataFrame,
    palette_light: list[str],
    palette_dark: list[str],
    out_path: Path,
    top_n: int = 30,
) -> None:
    """Generate a fully self-contained interactive bump chart HTML from the combined dataset.

    All topic and hour data are embedded as a compact JSON blob so the chart works
    offline and can be deployed to GitHub Pages without any backend.

    Features:
    - Dynamic pixel width: Math.max(containerW, hi.length*80) computed client-side at
      each redraw, so short ranges fill the viewport and horizontal scroll only appears
      when the range is wide enough to need it. No fixed width baked in at export time.
    - Boxed keyword labels at first+last non-NaN hour per topic via layout.annotations.
      Same declutter rule as render_bump_chart (both computed in Python before embedding).
    - Day-boundary markers: thick dashed vertical lines at each timezone's midnight
      crossing (UTC, GMT+8, GMT+7), with stacked annotations in format
      "{DayName}, DD MM YYYY ({tz})", recomputed on every range change.
    - Dual top+bottom x-axes: xaxis (bottom) + xaxis2 (overlaying='x', side='top').
      A hidden dummy scatter trace forces xaxis2 to render.
    - Single JS state model S={range,checked,pinned,hovered,dark} with one
      applyVis() as the only Plotly.restyle caller.
    - Sidebar search (#ssi) filters visible checkbox rows by substring match against
      topic display names -- does not touch chart opacity.
    - Time-range dropdown (Last 6h/24h/3d/7d/All) drives Plotly.react() client-side.
      rebuildCb(hi) only shows topics present in the current range.
    - Per-topic checkbox list for hard show/hide (independent of opacity dim).
    - Click-to-pin: click a trace to lock emphasis; click again to unpin.
    - Light/dark toggle; both palettes embedded; localStorage persistence (default dark).
    - Alte Haas Grotesk via fonts.cdnfonts.com CDN; fallback: 'Helvetica Neue', Arial.
      No local font files bundled; no new pip dependencies required.
    - Download CSV button (reconstructs CSV from embedded JSON blob).
    - Reset view button (clears all filters/pins/search).
    - Debounced sidebar search (150ms).
    - URL query-param sync (?range=7d) on every state change.
    - Topic stats readout in floating panel (appearances, best rank, total tweets).
    - ARIA labels on all interactive controls.
    - Keyboard nav: Escape clears pin/hover/stats.
    - Collapsible sidebar on mobile (hamburger toggle).
    """
    if df_combined is None or df_combined.empty:
        log.warning("df_combined is empty — skipping interactive chart render")
        return

    # ── 1. Pivot ──────────────────────────────────────────────────────────────
    pivot      = build_rank_pivot(df_combined, top_n)
    hour_cols  = list(pivot.columns)
    topic_list = list(pivot.index)
    n_hours    = len(hour_cols)
    n_topics   = len(topic_list)
    if n_hours == 0 or n_topics == 0:
        log.warning("Empty pivot — skipping interactive chart render")
        return

    # ── 2. Hour timestamps (for client-side range filtering) ──────────────────
    def _ts(lbl: str) -> int:
        try:
            core = lbl.split(" (")[0]  # strip " (Coordinated Universal Time)" etc.
            dt = datetime.strptime(core, "%a %b %d %Y %H:%M:%S GMT%z")
            return int(dt.timestamp())
        except Exception:
            log.warning("Could not parse hour_label as timestamp: %r", lbl)
            return 0

    hour_ts = [_ts(h) for h in hour_cols]
    max_ts   = max(hour_ts) if hour_ts else 0

    # ── 3. Topics JSON: colors pre-computed so JS needs no MD5 implementation ──
    tc_col = "tweet_count" if "tweet_count" in df_combined.columns else None
    topics_json = []
    for name_norm in topic_list:
        y_vals  = pivot.loc[name_norm].values
        cl      = get_topic_color(name_norm, palette_light)
        cd      = get_topic_color(name_norm, palette_dark)
        grp     = df_combined[df_combined["name_norm"] == name_norm]
        nm      = grp["name"].mode()
        display = str(nm.iloc[0]) if not nm.empty else name_norm
        bp_s    = grp["best_position"].dropna() if "best_position" in grp.columns else pd.Series(dtype="float64")
        tt_s    = grp["total_tweets"].dropna()  if "total_tweets"  in grp.columns else pd.Series(dtype="float64")
        bp      = int(bp_s.min()) if not bp_s.empty else None
        tt      = int(tt_s.max()) if not tt_s.empty else None
        pts = []
        for hi, (yv, hcol) in enumerate(zip(y_vals, hour_cols)):
            if yv is None or (isinstance(yv, float) and pd.isna(yv)):
                continue
            tc = None
            if tc_col:
                m = grp[grp["hour_label"] == hcol][tc_col]
                if not m.empty and pd.notna(m.iloc[0]):
                    try:
                        tc = int(m.iloc[0])
                    except (ValueError, TypeError):
                        tc = None
            pts.append([hi, int(yv), tc])
        topics_json.append({
            "n": name_norm, "d": display, "cl": cl, "cd": cd,
            "bp": bp, "tt": tt, "pts": pts,
        })

    # ── 4. Hours JSON (format_tz_label returns \n-separated strings; swap to <br>) ──
    hours_json = [
        {"l": format_tz_label(lbl).replace("\n", "<br>"), "ts": ts}
        for lbl, ts in zip(hour_cols, hour_ts)
    ]

    # ── 5. Serialize dataset ──────────────────────────────────────────────────
    dataset   = {"topN": top_n, "maxTs": max_ts, "hours": hours_json, "topics": topics_json}
    data_json = json.dumps(dataset, separators=(",", ":"), ensure_ascii=False)

    # ── 6. Embed Plotly.js bundle inline (offline-safe; no CDN dep in output) ──
    import plotly.offline as _plo
    plotly_js = f"<script>{_plo.get_plotlyjs()}</script>"

    # Inline the Lokanetra logo SVG so the exported HTML stays fully offline/self-contained.
    # The gradient fill (dark blue -> orange) reads fine on both light and dark backgrounds,
    # so one file covers both themes -- no separate light/dark asset needed.
    _logo_path = REPO_ROOT / "src" / "logo" / "lokentra.dev-logo.svg"
    logo_svg = _logo_path.read_text(encoding="utf-8").strip()
    logo_svg = re.sub(r'<svg\s+width="\d+"\s+height="\d+"', "<svg", logo_svg, count=1)

    # ── 7. Build HTML ─────────────────────────────────────────────────────────
    H = []
    H.append('<!DOCTYPE html><html lang="en">')
    H.append('<head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">')
    H.append('<title>Kestrel - Trending Submodule</title>')
    H.append('<link rel="preconnect" href="https://fonts.cdnfonts.com" crossorigin>')
    H.append('<link href="https://fonts.cdnfonts.com/css/alte-haas-grotesk" rel="stylesheet">')
    H.append(plotly_js)

    # CSS: must NOT be an f-string — CSS uses {} which Python would try to interpolate
    H.append('<style>')
    H.append(':root{--bg:#121212;--bg2:#1c1c1c;--fg:#e0e0e0;--fg2:#aaa;--bd:#333;'
             '--acc:#ff8c42;--ibg:#2a2a2a;--ibd:#444;--btn:#2a2a2a;--btnh:#3a3a3a;'
             '--st:#444;--sh:0 2px 12px rgba(0,0,0,.5)}')
    H.append('body.light{--bg:#fff;--bg2:#f5f5f5;--fg:#111;--fg2:#555;--bd:#ccc;'
             '--acc:#d4520a;--ibg:#fff;--ibd:#aaa;--btn:#eee;--btnh:#ddd;'
             '--st:#bbb;--sh:0 2px 12px rgba(0,0,0,.15)}')
    H.append('*{box-sizing:border-box;margin:0;padding:0}')
    H.append("body{font-family:'Alte Haas Grotesk','Helvetica Neue',Arial,sans-serif;"
             'background:var(--bg);color:var(--fg);display:flex;flex-direction:column;height:100vh;overflow:hidden}')
    H.append('#tb{display:flex;align-items:center;gap:8px;padding:6px 10px;'
             'background:var(--bg2);border-bottom:1px solid var(--bd);flex-shrink:0;flex-wrap:wrap}')
    H.append('#tb h1{font-size:14px;font-weight:400;margin-right:4px;white-space:nowrap}')
    H.append('#brand{display:flex;align-items:center;gap:8px;margin-right:4px}')
    H.append('#lka{display:flex;align-items:center;line-height:0}')
    H.append('#lka svg{height:20px;width:auto;display:block}')
    H.append('#rng{font-family:inherit;font-size:12px;background:var(--ibg);color:var(--fg);'
             'border:1px solid var(--ibd);border-radius:4px;padding:3px 6px;cursor:pointer}')
    H.append('.btn{font-family:inherit;font-size:12px;background:var(--btn);color:var(--fg);'
             'border:1px solid var(--bd);border-radius:4px;padding:3px 9px;cursor:pointer;transition:background .15s}')
    H.append('.btn:hover{background:var(--btnh)}')
    H.append('#tb .sp{flex:1}')
    H.append('#drng{font-size:12px;color:var(--fg2);white-space:nowrap}')
    H.append('#lay{display:flex;flex:1;overflow:hidden}')
    H.append('#sb{width:200px;flex-shrink:0;background:var(--bg2);border-right:1px solid var(--bd);'
             'display:flex;flex-direction:column;overflow:hidden;transition:width .2s}')
    H.append('#sb.cl{width:0;overflow:hidden}')
    H.append('#sbi{padding:8px;overflow-y:auto;flex:1}')
    H.append('#ssi{font-family:inherit;font-size:11px;background:var(--ibg);color:var(--fg);'
             'border:1px solid var(--ibd);border-radius:4px;padding:3px 7px;width:100%;margin-bottom:6px}')
    H.append('#ssi::placeholder{color:var(--fg2)}')
    H.append('#sbi h2{font-size:11px;text-transform:uppercase;letter-spacing:.06em;color:var(--fg2);margin-bottom:6px}')
    H.append('#sba{display:flex;gap:4px;margin:0 0 6px}')
    H.append('#sba .btn{font-size:10px;padding:2px 7px;flex:1}')
    H.append('.cr{display:flex;align-items:center;gap:5px;padding:2px 0;cursor:pointer}')
    H.append('.cr input[type=checkbox]{accent-color:var(--acc);cursor:pointer;width:13px;height:13px;flex-shrink:0}')
    H.append('.cr label{font-size:11px;color:var(--fg);cursor:pointer;overflow:hidden;'
             'text-overflow:ellipsis;white-space:nowrap;max-width:150px}')
    H.append('.csw{width:8px;height:8px;border-radius:50%;flex-shrink:0}')
    H.append('#ca{flex:1;display:flex;flex-direction:column;overflow:hidden}')
    H.append('#stp{display:none;position:fixed;right:12px;bottom:12px;z-index:100;'
             'background:var(--bg2);border:1px solid var(--bd);border-radius:6px;'
             'padding:10px 12px;font-size:12px;line-height:1.7;max-width:220px;box-shadow:var(--sh)}')
    H.append('#stp .stc{float:right;cursor:pointer;margin-left:8px;color:var(--fg2)}')
    H.append('#stp .stn{font-weight:700;font-size:13px;margin-bottom:4px;word-break:break-word}')
    H.append('#cr{display:flex;flex:1;overflow:hidden}')
    H.append('#yp{width:72px;flex-shrink:0;overflow:hidden;background:var(--bg);'
             'border-right:1px solid var(--bd);position:relative}')
    H.append('#yr{position:absolute;top:0;left:0;right:0;bottom:0}')
    H.append('.yl{font-size:10px;color:var(--fg2);text-align:right;line-height:1;'
             'position:absolute;right:4px;transform:translateY(-50%)}')
    H.append('#scr{flex:1;overflow-x:auto;overflow-y:hidden;background:var(--bg)}')
    H.append('#scr::-webkit-scrollbar{height:6px}')
    H.append('#scr::-webkit-scrollbar-thumb{background:var(--st);border-radius:3px}')
    H.append('#pd{min-width:100%}')
    H.append('@media(max-width:600px){#sb{position:absolute;z-index:200;height:100%;box-shadow:var(--sh)}}')
    H.append('#ftr{display:flex;align-items:center;gap:16px;padding:6px 12px;'
             'background:var(--bg2);border-top:1px solid var(--bd);flex-shrink:0;'
             'flex-wrap:wrap;font-size:11px;color:var(--fg2)}')
    H.append('#ftr p{margin:0}')
    H.append('#ftr a{color:var(--fg2);text-decoration:none}')
    H.append('#ftr a:hover{text-decoration:underline}')
    H.append('#ftr .cp{margin-left:auto}')
    H.append('</style>')
    H.append('<script>const _D=' + data_json + ';</script>')
    H.append('</head><body>')

    H.append(
        '<div id="tb" role="toolbar" aria-label="Chart controls">'
        '<button id="hb" class="btn" aria-label="Toggle sidebar" aria-expanded="true">&#9776;</button>'
        '<div id="brand"><h1>Kestrel - Trending Submodule</h1>'
        f'<a id="lka" href="https://lokanetra.dev/" target="_blank" rel="noopener" aria-label="Lokanetra">{logo_svg}</a></div>'
        '<label for="rng" style="font-size:12px">Range:</label>'
        '<select id="rng" aria-label="Time range">'
        '<option value="6h" selected>Last 6h</option>'
        '<option value="24h">Last 24h</option>'
        '<option value="3d">Last 3d</option>'
        '<option value="7d">Last 7d</option>'
        '<option value="All">All</option>'
        '</select>'
        '<button id="rst" class="btn" aria-label="Reset all filters">Reset view</button>'
        '<button id="dlb" class="btn" aria-label="Download combined CSV">Download CSV</button>'
        '<span id="drng"></span>'
        '<div class="sp"></div>'
        '<button id="db" class="btn" aria-label="Toggle light/dark mode">&#x2600;</button>'
        '</div>'
    )

    H.append(
        '<div id="lay">'
        '<div id="sb" role="complementary" aria-label="Topic visibility">'
        '<div id="sbi">'
        '<input id="ssi" type="search" placeholder="Search topics…" aria-label="Search topics" autocomplete="off">'
        '<h2>Topics</h2>'
        '<div id="sba">'
        '<button id="sall" class="btn" aria-label="Select all topics">All</button>'
        '<button id="snone" class="btn" aria-label="Deselect all topics">None</button>'
        '</div>'
        '<div id="cbl"></div></div>'
        '</div>'
        '<div id="ca">'
        '<div id="cr">'
        '<div id="yp" aria-hidden="true"><div id="yr"></div></div>'
        '<div id="scr"><div id="pd"></div></div>'
        '</div>'
        '</div>'
        '</div>'
        '<div id="stp" role="tooltip" aria-live="polite">'
        '<span id="stc" class="stc" title="Close">&#x2715;</span>'
        '<div class="stn" id="stn"></div><div id="stb"></div>'
        '</div>'
    )

    H.append(
        '<footer id="ftr">'
        '<p><a href="https://github.com/ishakmartins/kestrel-trending" target="_blank" rel="noopener">GitHub: kestrel-trending</a></p>'
        '<p>Cookies: <a href="https://lokanetra.dev/cookies" target="_blank" rel="noopener">https://lokanetra.dev/cookies</a></p>'
        '<p>Privacy: <a href="https://lokanetra.dev/privacy" target="_blank" rel="noopener">https://lokanetra.dev/privacy</a></p>'
        '<p>Terms: <a href="https://lokanetra.dev/terms" target="_blank" rel="noopener">https://lokanetra.dev/terms</a></p>'
        '<p>contact(at)lokanetra.dev</p>'
        '<p class="cp">&copy; 2026 Lokanetra. All rights reserved.</p>'
        '</footer>'
    )

    # JS block: raw string — width computed client-side, no __CW__ substitution needed.
    JS = r"""<script>
(function(){
"use strict";
const D=_D,N=D.topics.length;
const S={range:"6h",checked:{},pinned:null,hovered:null,dark:true};

// Init dark preference and URL params
(function(){
  const sd=localStorage.getItem("kestrel_dark");
  if(sd==="0")S.dark=false;
  if(!S.dark)document.body.classList.add("light");
  document.getElementById("db").textContent=S.dark?"☀":"🌙";
  const p=new URLSearchParams(location.search);
  if(p.has("range"))S.range=p.get("range");
  D.topics.forEach(t=>{S.checked[t.n]=true;});
})();

function syncUrl(){
  const p=new URLSearchParams();
  if(S.range!=="6h")p.set("range",S.range);
  history.replaceState(null,"",p.toString()?"?"+p:location.pathname);
}

function rngSec(r){return{"6h":21600,"24h":86400,"3d":259200,"7d":604800}[r]||null;}

function filtHi(){
  const s=rngSec(S.range);
  if(!s)return D.hours.map((_,i)=>i);
  const cut=D.maxTs-s;
  return D.hours.reduce((a,h,i)=>{if(h.ts>=cut)a.push(i);return a;},[]);
}

function buildTraces(hi){
  const hm={};hi.forEach((g,l)=>{hm[g]=l;});
  const tr=D.topics.map(t=>{
    const col=S.dark?t.cd:t.cl;
    const pts=t.pts.filter(p=>hm[p[0]]!==undefined);
    const xA=[],yA=[],tA=[];let pv=null;
    pts.forEach(p=>{
      const l=hm[p[0]];
      if(pv!==null&&l!==pv+1){xA.push(null);yA.push(null);tA.push("");}
      xA.push(l);yA.push(p[1]);
      tA.push("<b>"+t.d+"</b><br>Rank "+p[1]+(p[2]!=null?" · "+p[2].toLocaleString()+" tweets":""));
      pv=l;
    });
    return{type:"scatter",mode:"lines",name:t.d,x:xA,y:yA,text:tA,
      hovertemplate:"%{text}<extra></extra>",
      line:{color:col,width:1.8},opacity:1,connectgaps:false,showlegend:false};
  });
  // Dummy trace: forces xaxis2 (top x-axis) to render in Plotly
  tr.push({type:"scatter",mode:"markers",x:[0],y:[null],xaxis:"x2",
    showlegend:false,hoverinfo:"skip",marker:{size:0,opacity:0}});
  return tr;
}

function buildLayout(hi){
  const tv=hi.map((_,l)=>l);
  const tt=hi.map(i=>D.hours[i].l.replace(/<br>/g,"\n"));
  const d=S.dark,bg=d?"#121212":"#fff",fg=d?"#e0e0e0":"#111";
  const grid=d?"#2a2a2a":"#ccc",gridx=d?"#383838":"#999";
  const tf={family:"'Alte Haas Grotesk','Helvetica Neue',Arial,sans-serif",size:9};
  const axC={tickmode:"array",tickvals:tv,ticktext:tt,tickfont:tf,showline:false,zeroline:false,automargin:true};
  const perHourPx=80;
  const containerW=document.getElementById("scr").clientWidth||1200;
  const dynW=Math.max(containerW,hi.length*perHourPx);
  return{
    width:dynW,height:560,
    paper_bgcolor:bg,plot_bgcolor:bg,
    font:{family:"'Alte Haas Grotesk','Helvetica Neue',Arial,sans-serif",color:fg,size:11},
    margin:{l:5,r:10,t:40,b:80},
    xaxis:Object.assign({},axC,{showgrid:true,gridcolor:gridx,gridwidth:1,side:"bottom"}),
    xaxis2:Object.assign({},axC,{overlaying:"x",side:"top",matches:"x",showgrid:false}),
    yaxis:{range:[D.topN+0.5,0.5],showticklabels:false,showgrid:true,
      gridcolor:grid,gridwidth:1,zeroline:false,showline:false},
    hovermode:"closest",dragmode:"pan",
  };
}

function buildAnns(hi){
  const hm={};hi.forEach((g,l)=>{hm[g]=l;});
  const anns=[];
  D.topics.forEach(t=>{
    if(S.checked[t.n]===false)return;
    const col=S.dark?t.cd:t.cl,bx=S.dark?"#1c1c1c":"#fff";
    const pts=t.pts.filter(p=>hm[p[0]]!==undefined);
    if(!pts.length)return;
    // First and last pts only — same rule as render_bump_chart static renderer
    (pts.length===1?[pts[0]]:[pts[0],pts[pts.length-1]]).forEach(p=>{
      anns.push({x:hm[p[0]],y:p[1],text:t.d,showarrow:false,
        font:{family:"'Alte Haas Grotesk','Helvetica Neue',Arial,sans-serif",size:10,color:col},
        bgcolor:bx,bordercolor:col,borderwidth:1,borderpad:3,opacity:0.93,
        xanchor:"center",yanchor:"middle"});
    });
  });
  return anns;
}

const DAYS=["Sunday","Monday","Tuesday","Wednesday","Thursday","Friday","Saturday"];
const MONS=["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"];
const tzList=[{name:"UTC",off:0},{name:"GMT+8",off:28800},{name:"GMT+7",off:25200}];
function fmtTs(epochSec,off){
  const dt=new Date((epochSec+off)*1000);
  return{
    dayName:DAYS[dt.getUTCDay()],
    dd:String(dt.getUTCDate()).padStart(2,"0"),
    mon:MONS[dt.getUTCMonth()],
    mm:String(dt.getUTCMonth()+1).padStart(2,"0"),
    yyyy:String(dt.getUTCFullYear()),
    hhmm:String(dt.getUTCHours()).padStart(2,"0")+":"+String(dt.getUTCMinutes()).padStart(2,"0")
  };
}

function buildDayBoundaries(hi){
  const dk=S.dark;
  const fc=dk?"#888":"#666",bc=dk?"#1c1c1c":"#fff",lc=dk?"#555":"#aaa";
  // Map local position in hi -> array of label strings
  const byPos={};
  tzList.forEach(tz=>{
    let prevDate=null;
    hi.forEach((g,l)=>{
      const f=fmtTs(D.hours[g].ts,tz.off);
      const date=Number(f.yyyy)*10000+Number(f.mm)*100+Number(f.dd);
      if(prevDate!==null&&date!==prevDate){
        if(!byPos[l])byPos[l]=[];
        byPos[l].push(f.dayName+", "+f.dd+" "+f.mm+" "+f.yyyy+" ("+tz.name+")");
      }
      prevDate=date;
    });
  });
  // Group adjacent positions (within 1 column) so nearby boundaries share one line
  const positions=Object.keys(byPos).map(Number).sort((a,b)=>a-b);
  const groups=[];
  positions.forEach(pos=>{
    if(groups.length===0||pos>groups[groups.length-1].pos+1){
      groups.push({pos,labels:[...byPos[pos]]});
    } else {
      groups[groups.length-1].labels.push(...byPos[pos]);
    }
  });
  const shapes=[],anns=[];
  groups.forEach(({pos,labels})=>{
    shapes.push({
      type:"line",x0:pos-0.5,x1:pos-0.5,y0:0,y1:1,yref:"paper",
      line:{color:lc,width:3,dash:"dash"}
    });
    anns.push({
      x:pos-0.5,y:0.98,yref:"paper",
      text:labels.join("<br>"),
      showarrow:false,xanchor:"left",yanchor:"top",
      font:{family:"'Alte Haas Grotesk','Helvetica Neue',Arial,sans-serif",size:8,color:fc},
      bgcolor:bc,bordercolor:lc,borderwidth:1,borderpad:2,opacity:0.9,xshift:3
    });
  });
  return{shapes,anns};
}

// Single Plotly.restyle caller — reads all state to compute final opacity per trace.
// Fixes v6 bug: plotly_unhover called restyle independently and wiped search filter.
function applyVis(){
  const gd=document.getElementById("pd");
  if(!gd.data)return;
  const op=D.topics.map(t=>{
    const chk=S.checked[t.n]!==false;
    if(!chk)return 0;
    if(S.pinned)return S.pinned===t.n?1:0.06;
    if(S.hovered)return S.hovered===t.n?1:0.06;
    return 1;
  });
  op.push(0); // dummy trace stays invisible
  Plotly.restyle(gd,{opacity:op},Array.from({length:op.length},(_,i)=>i));
  if(S.hi){
    Plotly.relayout(gd,{annotations:[...buildAnns(S.hi),...S.dbAnns]});
  }
}

function updYp(){
  const gd=document.getElementById("pd");
  if(!gd._fullLayout)return;
  const fl=gd._fullLayout,m=fl.margin,pH=fl.height-m.t-m.b;
  const yr=fl.yaxis.range;
  // yr[0]=bottom screen value (large rank), yr[1]=top screen value (small rank)
  const yBig=Math.max(yr[0],yr[1]),ySm=Math.min(yr[0],yr[1]);
  const panel=document.getElementById("yr");
  panel.innerHTML="";
  for(let r=1;r<=D.topN;r++){
    if(r<ySm-0.5||r>yBig+0.5)continue;
    const el=document.createElement("div");
    el.className="yl";el.textContent=r;
    el.style.top=(m.t+(r-ySm)/(yBig-ySm)*pH)+"px";
    panel.appendChild(el);
  }
}

function rebuildCb(hi){
  const hiSet=new Set(hi);
  const list=document.getElementById("cbl");
  list.innerHTML="";
  D.topics.forEach(t=>{
    if(!t.pts.some(p=>hiSet.has(p[0])))return;
    const col=S.dark?t.cd:t.cl;
    const row=document.createElement("div");row.className="cr";
    const sw=document.createElement("div");sw.className="csw";sw.style.background=col;
    const cb=document.createElement("input");cb.type="checkbox";
    cb.checked=S.checked[t.n]!==false;
    cb.setAttribute("aria-label","Show "+t.d);
    cb.addEventListener("change",()=>{S.checked[t.n]=cb.checked;applyVis();});
    const lb=document.createElement("label");lb.textContent=t.d;lb.title=t.d;
    lb.addEventListener("click",()=>{cb.click();});
    row.appendChild(sw);row.appendChild(cb);row.appendChild(lb);
    list.appendChild(row);
  });
  // Re-apply current sidebar search filter after rebuild
  const sv=document.getElementById("ssi").value.trim().toLowerCase();
  if(sv){
    list.querySelectorAll(".cr").forEach(row=>{
      const lb=row.querySelector("label");
      row.style.display=(lb&&lb.textContent.toLowerCase().includes(sv))?"":"none";
    });
  }
}

let _first=true;
function updDrng(hi){
  const el=document.getElementById("drng");
  if(!hi.length){el.textContent="Date range of displayed data: (no data in range)";return;}
  function fmt(ts){
    const f0=fmtTs(ts,0);
    const parts=tzList.map(tz=>tz.name+": "+fmtTs(ts,tz.off).hhmm);
    return "["+f0.dayName+", "+f0.dd+" "+f0.mon+" "+f0.yyyy+", "+parts.join(", ")+"]";
  }
  el.textContent="Date range of displayed data: "+fmt(D.hours[hi[0]].ts)+" to "+fmt(D.hours[hi[hi.length-1]].ts);
}

function redraw(){
  const gd=document.getElementById("pd");
  const hi=filtHi(),tr=buildTraces(hi),ly=buildLayout(hi);
  updDrng(hi);
  const db=buildDayBoundaries(hi);
  ly.shapes=db.shapes;
  ly.annotations=[...buildAnns(hi),...db.anns];
  S.hi=hi;S.dbAnns=db.anns;
  (_first?Plotly.newPlot:Plotly.react)(gd,tr,ly,{responsive:false,displayModeBar:true,scrollZoom:true})
    .then(()=>{if(_first){attEv(gd);_first=false;}updYp();rebuildCb(hi);applyVis();syncUrl();});
}

function attEv(gd){
  gd.on("plotly_hover",d=>{
    if(!d.points||!d.points[0])return;
    const ti=d.points[0].curveNumber;
    if(ti>=N)return; // ignore dummy trace
    S.hovered=D.topics[ti].n;shSt(S.hovered);applyVis();
  });
  gd.on("plotly_unhover",()=>{
    S.hovered=null;
    if(!S.pinned)document.getElementById("stp").style.display="none";
    applyVis();
  });
  gd.on("plotly_click",d=>{
    if(!d.points||!d.points[0])return;
    const ti=d.points[0].curveNumber;
    if(ti>=N)return;
    const kn=D.topics[ti].n;
    S.pinned=(S.pinned===kn)?null:kn;
    if(S.pinned)shSt(kn);
    else document.getElementById("stp").style.display="none";
    applyVis();
  });
  gd.on("plotly_relayout",()=>{updYp();});
}

function shSt(kn){
  const t=D.topics.find(x=>x.n===kn);if(!t)return;
  document.getElementById("stn").textContent=t.d;
  let b="<div>Appearances: "+t.pts.length+" / "+D.hours.length+" hrs</div>";
  if(t.pts.length)b+="<div>Best rank: #"+Math.min(...t.pts.map(p=>p[1]))+"</div>";
  if(t.bp!=null)b+="<div>Best position: #"+t.bp+"</div>";
  if(t.tt!=null)b+="<div>Total tweets: "+t.tt.toLocaleString()+"</div>";
  document.getElementById("stb").innerHTML=b;
  document.getElementById("stp").style.display="block";
}
function clSt(){document.getElementById("stp").style.display="none";S.pinned=null;applyVis();}

function setRange(v){S.range=v;redraw();}

let _sb=null;
function sbSch(){
  clearTimeout(_sb);
  _sb=setTimeout(()=>{
    const v=document.getElementById("ssi").value.trim().toLowerCase();
    document.querySelectorAll("#cbl .cr").forEach(row=>{
      const lb=row.querySelector("label");
      row.style.display=(!v||(lb&&lb.textContent.toLowerCase().includes(v)))?"":"none";
    });
  },150);
}

function rstV(){
  S.range="All";S.pinned=null;S.hovered=null;
  D.topics.forEach(t=>{S.checked[t.n]=true;});
  document.getElementById("ssi").value="";
  document.querySelectorAll("#cbl .cr").forEach(row=>{row.style.display="";});
  document.getElementById("rng").value="All";
  document.getElementById("stp").style.display="none";
  redraw();
}

function dlCsv(){
  const rows=["name_norm,display,hour_label,hour_ts,rank,tweet_count"];
  D.topics.forEach(t=>{
    t.pts.forEach(p=>{
      const h=D.hours[p[0]];
      rows.push([t.n,t.d,h.l.replace(/<br>/g," | "),h.ts,p[1],p[2]!=null?p[2]:""].join(","));
    });
  });
  const bl=new Blob([rows.join("\n")],{type:"text/csv"});
  const u=URL.createObjectURL(bl),a=document.createElement("a");
  a.href=u;a.download="kestrel_trends_combined.csv";a.click();URL.revokeObjectURL(u);
}

function tDk(){
  S.dark=!S.dark;
  document.body.classList.toggle("light",!S.dark);
  document.getElementById("db").textContent=S.dark?"☀":"🌙";
  localStorage.setItem("kestrel_dark",S.dark?"1":"0");
  redraw();
}

function tSb(){
  const s=document.getElementById("sb");
  s.classList.toggle("cl");
  document.getElementById("hb").setAttribute("aria-expanded",s.classList.contains("cl")?"false":"true");
}

document.addEventListener("keydown",e=>{
  if(e.key==="Escape"){S.pinned=null;S.hovered=null;document.getElementById("stp").style.display="none";applyVis();}
});

// Wire all controls via addEventListener — keeps handlers closure-scoped, not global
document.getElementById("hb").addEventListener("click", tSb);
document.getElementById("rng").addEventListener("change", e=>setRange(e.target.value));
document.getElementById("ssi").addEventListener("input", sbSch);
document.getElementById("rst").addEventListener("click", rstV);
document.getElementById("dlb").addEventListener("click", dlCsv);
document.getElementById("db").addEventListener("click", tDk);
document.getElementById("stc").addEventListener("click", clSt);
document.getElementById("sall").addEventListener("click",()=>{D.topics.forEach(t=>{S.checked[t.n]=true;});document.querySelectorAll("#cbl input[type=checkbox]").forEach(cb=>{cb.checked=true;});applyVis();});
document.getElementById("snone").addEventListener("click",()=>{D.topics.forEach(t=>{S.checked[t.n]=false;});document.querySelectorAll("#cbl input[type=checkbox]").forEach(cb=>{cb.checked=false;});applyVis();});
// Restore URL params to UI controls before first draw
document.getElementById("rng").value=S.range;
redraw();
})();
</script>"""
    H.append(JS)
    H.append('</body></html>')

    html = "\n".join(H)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(html, encoding="utf-8")
    log.info(
        "Interactive chart saved: %s (%d topics, %d hour-columns, ~%d KB)",
        out_path, n_topics, n_hours, len(html.encode("utf-8")) // 1024,
    )


In [12]:
def refresh_latest_pointers(run_dir: Path, run_id: str, output_root: Path) -> None:
    """Copy run outputs to stable latest_* paths outside the timestamped run folder.

    Gives a GitHub Pages site (or any other consumer) a single unchanging URL to
    point at instead of discovering the newest run_<id>/ folder each time.
    Safe to call repeatedly; shutil.copy2 overwrites the destination if it exists.
    """
    mapping = [
        (
            run_dir / f"trends24_id_bumpchart_{run_id}.png",
            output_root / "latest_light.png",
        ),
        (
            run_dir / f"trends24_id_bumpchart_{run_id}_dark.png",
            output_root / "latest_dark.png",
        ),
        (
            run_dir / f"trends24_id_bumpchart_{run_id}_interactive.html",
            output_root / "latest_interactive.html",
        ),
    ]
    for src, dst in mapping:
        if src.exists():
            shutil.copy2(src, dst)
            log.info("Latest pointer refreshed: %s -> %s", src.name, dst.name)
        else:
            log.warning("Source not found, skipped latest pointer: %s", src)

In [13]:
def build_combined_csv(output_root: Path) -> pd.DataFrame | None:
    """Rebuild the combined CSV from all run_* CSVs under output_root.

    Reads every trends24_id_raw_*.csv in run_*/ subdirectories, concatenates them,
    then deduplicates on (name_norm, hour_label). When the same pair appears in
    multiple runs, the row from the run with the latest captured_at_utc is kept.
    Rationale: later captures are more authoritative — the site may have corrected or
    backfilled data since the earlier run, so the newest observation wins.

    Rebuilds from scratch on every call (full re-glob + re-dedupe). At current data
    volumes this is fast; an incremental-append path is not needed yet.

    Returns the combined DataFrame (also written to output_root/combined/), or None if
    no run CSVs are found under output_root.
    """
    run_csvs = sorted(output_root.glob("run_*/trends24_id_raw_*.csv"))
    if not run_csvs:
        log.warning("No run CSVs found under %s — combined CSV not built", output_root)
        return None

    frames = []
    for p in run_csvs:
        try:
            frames.append(pd.read_csv(p, encoding="utf-8"))
        except Exception as e:
            log.warning("Skipping %s (%s)", p.name, e)

    if not frames:
        log.warning("All run CSVs failed to load — combined CSV not built")
        return None

    combined = pd.concat(frames, ignore_index=True)
    total_before = len(combined)

    # Dedupe: sort descending by captured_at_utc so the latest row appears first,
    # then drop_duplicates(keep='first') retains the latest for each (name_norm, hour_label).
    combined["captured_at_utc"] = pd.to_datetime(
        combined["captured_at_utc"], utc=True, errors="coerce"
    )
    combined = combined.sort_values("captured_at_utc", ascending=False)
    combined = combined.drop_duplicates(subset=["name_norm", "hour_label"], keep="first")
    combined = combined.sort_values(["hour_label", "rank"]).reset_index(drop=True)

    out_dir = output_root / "combined"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "trends24_id_combined.csv"
    combined.to_csv(out_path, index=False, encoding="utf-8")
    log.info(
        "Combined CSV: %s | %d rows from %d run(s), %d after dedupe on (name_norm, hour_label)",
        out_path, total_before, len(run_csvs), len(combined),
    )
    return combined

## Part B - Config

Constants block and precomputed Pumpkin palette. All runtime-tunable values live here.
Nothing below should need changing for a normal run; adjust selectors in Part A if
markup has shifted.

In [14]:
# --- Target URL ---
TRENDS24_URL = "https://trends24.in/indonesia/"

# --- Output root ---
# Anchored to repo root so the path resolves the same way regardless of Jupyter's cwd.
# In v1–v6, OUTPUT_ROOT = Path("dataset/trends") resolved to notebooks/dataset/trends/
# when Jupyter was launched from the notebooks/ directory, and to repo_root/dataset/trends/
# when launched from the repo root. This function walks up from cwd until it finds the
# .git marker and anchors OUTPUT_ROOT there, eliminating the ambiguity.
def _find_repo_root() -> Path:
    """Walk up from cwd until .git is found. Falls back to cwd if not found within 6 levels."""
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / ".git").exists():
            return p
        p = p.parent
    return Path.cwd().resolve()

REPO_ROOT   = _find_repo_root()
OUTPUT_ROOT = REPO_ROOT / "dataset" / "trends"

# --- HTTP ---
# Honest, non-spoofed UA - trends24.in robots.txt permits generic User-agent: * .
# Do not impersonate a browser UA; that would misrepresent the request.
USER_AGENT = (
    "Mozilla/5.0 (compatible; kestrel-trend-capture/1.0; "
    "+https://github.com/rednosereindeer80/kestrel-trending)"
)
REQUEST_TIMEOUT = 20  # seconds

# --- Schema-drift sanity guards ---
# parse_timeline_columns raises if the largest visible column has fewer trends than this.
# Verified range on trends24.in is ~50; 30 is a conservative floor (fail loud).
MIN_EXPECTED_TRENDS = 30

# --- Palette size ---
# Must match build_pumpkin_palette()'s internal 6×5×5 grid. Change both together.
PALETTE_SIZE = 150

# --- Chart display ---
# Number of top-ranked topics to display in the bump chart.
# Supported values: 25, 30, 50. Change only this constant; no other cell needs editing.
TOP_N = 50

# --- Output filename suffixes ---
PNG_DARK_SUFFIX = "_dark"
HTML_SUFFIX     = "_interactive"

In [15]:
PUMPKIN_PALETTE = build_pumpkin_palette(PALETTE_SIZE)
print(f"Palette:      {len(PUMPKIN_PALETTE)} colors | hue 4°–69° | sat 70–100% | lit 35–65%")
print(f"Anchors:      {PUMPKIN_PALETTE[0]}  {PUMPKIN_PALETTE[37]}  {PUMPKIN_PALETTE[74]}  {PUMPKIN_PALETTE[111]}  {PUMPKIN_PALETTE[149]}")

PUMPKIN_PALETTE_DARK = build_pumpkin_palette_dark(PALETTE_SIZE)
print(f"Palette dark: {len(PUMPKIN_PALETTE_DARK)} colors | hue 4°–69° | sat 70–100% | lit 55–82%")
print(f"Anchors dark: {PUMPKIN_PALETTE_DARK[0]}  {PUMPKIN_PALETTE_DARK[37]}  {PUMPKIN_PALETTE_DARK[74]}  {PUMPKIN_PALETTE_DARK[111]}  {PUMPKIN_PALETTE_DARK[149]}")

Palette:      150 colors | hue 4°–69° | sat 70–100% | lit 35–65%
Anchors:      #97231a  #eb5013  #fea54c  #c8bc10  #e4fe4c
Palette dark: 150 colors | hue 4°–69° | sat 70–100% | lit 55–82%
Anchors dark: #dc463b  #f2916a  #fed1a3  #f0e54a  #f1fea3


## Part C - Orchestration

One cell per step. Run top-to-bottom ("Run All") with no manual steps and no
required environment variables.

**Single-request design:** the page is fetched once in C2. Both the timeline parser
(C3) and detail-stats parser (C4) operate on the same `soup` object - only 1 HTTP
request total regardless of how many trends or hour-columns are visible.

In [16]:
# Compute run timestamp ONCE here, UTC. Reused for folder name, file names, run_id column.
# Never re-compute inside later cells — avoids folder/file stamp drift across slow cells.
RUN_TS = datetime.now(timezone.utc)
RUN_ID = RUN_TS.strftime("%Y%m%d-%H%M%S")
RUN_DIR = OUTPUT_ROOT / f"run_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH          = RUN_DIR / f"trends24_id_raw_{RUN_ID}.csv"
PNG_PATH          = RUN_DIR / f"trends24_id_bumpchart_{RUN_ID}.png"
PNG_DARK_PATH     = RUN_DIR / f"trends24_id_bumpchart_{RUN_ID}{PNG_DARK_SUFFIX}.png"
HTML_PATH         = RUN_DIR / f"trends24_id_bumpchart_{RUN_ID}{HTML_SUFFIX}.html"
COMBINED_CSV_PATH = OUTPUT_ROOT / "combined" / "trends24_id_combined.csv"

log.info("run_id=%s  output_dir=%s  repo_root=%s", RUN_ID, RUN_DIR, REPO_ROOT)

22:07:04 [INFO] run_id=20260721-140704  output_dir=X:\githubrepo\kestrel-trending\dataset\trends\run_20260721-140704  repo_root=X:\githubrepo\kestrel-trending


In [17]:
log.info("Fetching %s ...", TRENDS24_URL)
soup = fetch_page(TRENDS24_URL, USER_AGENT, REQUEST_TIMEOUT)
log.info("Fetch OK")

22:07:04 [INFO] Fetching https://trends24.in/indonesia/ ...
22:07:05 [INFO] Fetch OK


In [18]:
log.info("Parsing timeline columns ...")
timeline_columns = parse_timeline_columns(soup, min_trends=MIN_EXPECTED_TRENDS)

log.info(
    "Parsed %d hour-column(s): %s",
    len(timeline_columns),
    [c["hour_label"] for c in timeline_columns],
)
log.info(
    "Trends per column: %s",
    [len(c["trends"]) for c in timeline_columns],
)

22:07:05 [INFO] Parsing timeline columns ...
22:07:05 [INFO] Parsed 24 hour-column(s): ['Tue Jul 21 2026 13:19:02 GMT+0000 (Coordinated Universal Time)', 'Tue Jul 21 2026 12:28:07 GMT+0000 (Coordinated Universal Time)', 'Tue Jul 21 2026 12:28:05 GMT+0000 (Coordinated Universal Time)', 'Tue Jul 21 2026 11:36:14 GMT+0000 (Coordinated Universal Time)', 'Tue Jul 21 2026 11:36:13 GMT+0000 (Coordinated Universal Time)', 'Tue Jul 21 2026 10:44:13 GMT+0000 (Coordinated Universal Time)', 'Tue Jul 21 2026 09:52:13 GMT+0000 (Coordinated Universal Time)', 'Tue Jul 21 2026 09:00:57 GMT+0000 (Coordinated Universal Time)', 'Tue Jul 21 2026 08:09:48 GMT+0000 (Coordinated Universal Time)', 'Tue Jul 21 2026 07:18:55 GMT+0000 (Coordinated Universal Time)', 'Tue Jul 21 2026 06:27:53 GMT+0000 (Coordinated Universal Time)', 'Tue Jul 21 2026 05:36:13 GMT+0000 (Coordinated Universal Time)', 'Tue Jul 21 2026 05:36:11 GMT+0000 (Coordinated Universal Time)', 'Tue Jul 21 2026 04:44:10 GMT+0000 (Coordinated Univer

In [19]:
log.info("Checking for detail-stats table in raw HTML ...")
detail_stats = parse_detail_stats_table(soup)

if detail_stats is None:
    log.warning(
        "Detail-stats table absent from raw HTML. "
        "Possible reasons: (a) the /indonesia/ subpage does not have the Table tab "
        "that the worldwide page has - check in a browser; "
        "(b) the table is JS-injected rather than pre-rendered - if so, install "
        "playwright (pip install playwright && playwright install chromium) and "
        "re-fetch with a headless browser. "
        "Proceeding with timeline data only; best_position / total_tweets / "
        "trending_for_* will be null in the output CSV."
    )
    detail_stats = []
else:
    log.info("Detail-stats table found in raw HTML: %d rows parsed", len(detail_stats))

22:07:05 [INFO] Checking for detail-stats table in raw HTML ...
22:07:05 [WARNING] Detail-stats table absent from raw HTML. Possible reasons: (a) the /indonesia/ subpage does not have the Table tab that the worldwide page has - check in a browser; (b) the table is JS-injected rather than pre-rendered - if so, install playwright (pip install playwright && playwright install chromium) and re-fetch with a headless browser. Proceeding with timeline data only; best_position / total_tweets / trending_for_* will be null in the output CSV.


In [20]:
log.info("Normalising + joining on name_norm ...")
df = join_data(timeline_columns, detail_stats, RUN_TS, RUN_ID, TRENDS24_URL)

total_rows = len(df)
unique_topics = df["name_norm"].nunique()
null_join_count = int(df["best_position"].isna().sum())

log.info(
    "DataFrame: %d rows | %d unique topics | %d rows missing detail-stats data",
    total_rows, unique_topics, null_join_count,
)
df.head(10)

22:07:05 [INFO] Normalising + joining on name_norm ...
22:07:05 [INFO] DataFrame: 1197 rows | 147 unique topics | 1197 rows missing detail-stats data


,captured_at_utc,run_id,source_url,hour_label,rank,name,name_norm,tweet_count_raw,tweet_count,best_position,total_tweets,trending_for_raw,trending_for_hours
0,2026-07-21T14:07:04.852778+00:00,20260721-140704,https://trends24.in/indonesia/,Tue Jul 21 2026 13:19:02 GMT+0000 (Coordinated...,1,#MrKillSeriesEP3,mrkillseriesep3,None,None,None,None,None,None
1,2026-07-21T14:07:04.852778+00:00,20260721-140704,https://trends24.in/indonesia/,Tue Jul 21 2026 13:19:02 GMT+0000 (Coordinated...,2,Samsung Galaxy Fold 8,samsung galaxy fold 8,None,None,None,None,None,None
2,2026-07-21T14:07:04.852778+00:00,20260721-140704,https://trends24.in/indonesia/,Tue Jul 21 2026 13:19:02 GMT+0000 (Coordinated...,3,#SkyNanixCarà¸£à¸²Carà¸à¸±à¸,skynanixcarà¸£à¸²carà¸à¸±à¸,None,None,None,None,None,None
3,2026-07-21T14:07:04.852778+00:00,20260721-140704,https://trends24.in/indonesia/,Tue Jul 21 2026 13:19:02 GMT+0000 (Coordinated...,4,#Flex1045xWilliamjkp_Flashback,flex1045xwilliamjkp_flashback,None,None,None,None,None,None
4,2026-07-21T14:07:04.852778+00:00,20260721-140704,https://trends24.in/indonesia/,Tue Jul 21 2026 13:19:02 GMT+0000 (Coordinated...,5,WILLIAM SOLO PRESS,william solo press,None,None,None,None,None,None
5,2026-07-21T14:07:04.852778+00:00,20260721-140704,https://trends24.in/indonesia/,Tue Jul 21 2026 13:19:02 GMT+0000 (Coordinated...,6,#FlashbackWilliamjkp_Presstour,flashbackwilliamjkp_presstour,None,None,None,None,None,None
6,2026-07-21T14:07:04.852778+00:00,20260721-140704,https://trends24.in/indonesia/,Tue Jul 21 2026 13:19:02 GMT+0000 (Coordinated...,7,#EffaclarxPondNaravit,effaclarxpondnaravit,None,None,None,None,None,None
7,2026-07-21T14:07:04.852778+00:00,20260721-140704,https://trends24.in/indonesia/,Tue Jul 21 2026 13:19:02 GMT+0000 (Coordinated...,8,LRP EFFACLAR WITH POND,lrp effaclar with pond,None,None,None,None,None,None
8,2026-07-21T14:07:04.852778+00:00,20260721-140704,https://trends24.in/indonesia/,Tue Jul 21 2026 13:19:02 GMT+0000 (Coordinated...,9,ELIOT UNFOLDING LEGACY,eliot unfolding legacy,None,None,None,None,None,None
9,2026-07-21T14:07:04.852778+00:00,20260721-140704,https://trends24.in/indonesia/,Tue Jul 21 2026 13:19:02 GMT+0000 (Coordinated...,10,CURHAT SENADA,curhat senada,None,None,None,None,None,None


In [21]:
write_raw_csv(df, CSV_PATH)

22:07:05 [INFO] CSV written: X:\githubrepo\kestrel-trending\dataset\trends\run_20260721-140704\trends24_id_raw_20260721-140704.csv (1197 rows)


In [22]:
log.info("Rebuilding combined CSV from all runs under %s ...", OUTPUT_ROOT)
df_combined = build_combined_csv(OUTPUT_ROOT)
if df_combined is None:
    log.warning("Combined CSV build failed — falling back to current run data for interactive chart")
    df_combined = df
else:
    log.info(
        "Combined dataset ready: %d rows, %d unique topics",
        len(df_combined), df_combined["name_norm"].nunique(),
    )

22:07:05 [INFO] Rebuilding combined CSV from all runs under X:\githubrepo\kestrel-trending\dataset\trends ...
22:07:05 [INFO] Combined CSV: X:\githubrepo\kestrel-trending\dataset\trends\combined\trends24_id_combined.csv | 7150 rows from 6 run(s), 2575 after dedupe on (name_norm, hour_label)
22:07:05 [INFO] Combined dataset ready: 2575 rows, 311 unique topics


In [23]:
log.info("Rendering bump chart ...")
render_bump_chart(df, PUMPKIN_PALETTE, PNG_PATH, top_n=TOP_N)


22:07:05 [INFO] Rendering bump chart ...
C:\Users\hapos\AppData\Local\Temp\ipykernel_27284\2515180816.py:146: UserWarning: Glyph 139 (\x8b) missing from font(s) DejaVu Sans.
  fig.savefig(out_path, dpi=150, bbox_inches="tight",
C:\Users\hapos\AppData\Local\Temp\ipykernel_27284\2515180816.py:146: UserWarning: Glyph 135 (\x87) missing from font(s) DejaVu Sans.
  fig.savefig(out_path, dpi=150, bbox_inches="tight",
C:\Users\hapos\AppData\Local\Temp\ipykernel_27284\2515180816.py:146: UserWarning: Glyph 129 (\x81) missing from font(s) DejaVu Sans.
  fig.savefig(out_path, dpi=150, bbox_inches="tight",
C:\Users\hapos\AppData\Local\Temp\ipykernel_27284\2515180816.py:146: UserWarning: Glyph 149 (\x95) missing from font(s) DejaVu Sans.
  fig.savefig(out_path, dpi=150, bbox_inches="tight",
22:07:12 [INFO] Bump chart saved: X:\githubrepo\kestrel-trending\dataset\trends\run_20260721-140704\trends24_id_bumpchart_20260721-140704.png (147 topics, 24 hour-columns, dark=False)


In [24]:
log.info("Rendering dark bump chart ...")
render_bump_chart(df, PUMPKIN_PALETTE_DARK, PNG_DARK_PATH, top_n=TOP_N, dark=True)

22:07:12 [INFO] Rendering dark bump chart ...
C:\Users\hapos\AppData\Local\Temp\ipykernel_27284\2515180816.py:146: UserWarning: Glyph 139 (\x8b) missing from font(s) DejaVu Sans.
  fig.savefig(out_path, dpi=150, bbox_inches="tight",
C:\Users\hapos\AppData\Local\Temp\ipykernel_27284\2515180816.py:146: UserWarning: Glyph 135 (\x87) missing from font(s) DejaVu Sans.
  fig.savefig(out_path, dpi=150, bbox_inches="tight",
C:\Users\hapos\AppData\Local\Temp\ipykernel_27284\2515180816.py:146: UserWarning: Glyph 129 (\x81) missing from font(s) DejaVu Sans.
  fig.savefig(out_path, dpi=150, bbox_inches="tight",
C:\Users\hapos\AppData\Local\Temp\ipykernel_27284\2515180816.py:146: UserWarning: Glyph 149 (\x95) missing from font(s) DejaVu Sans.
  fig.savefig(out_path, dpi=150, bbox_inches="tight",
22:07:18 [INFO] Bump chart saved: X:\githubrepo\kestrel-trending\dataset\trends\run_20260721-140704\trends24_id_bumpchart_20260721-140704_dark.png (147 topics, 24 hour-columns, dark=True)


In [25]:
log.info("Rendering interactive chart (combined dataset: %d rows) ...", len(df_combined))
render_bump_chart_interactive(
    df_combined, PUMPKIN_PALETTE, PUMPKIN_PALETTE_DARK, HTML_PATH, top_n=TOP_N
)

22:07:18 [INFO] Rendering interactive chart (combined dataset: 2575 rows) ...
22:07:21 [INFO] Interactive chart saved: X:\githubrepo\kestrel-trending\dataset\trends\run_20260721-140704\trends24_id_bumpchart_20260721-140704_interactive.html (311 topics, 52 hour-columns, ~4827 KB)


In [26]:
log.info("Refreshing latest pointers ...")
refresh_latest_pointers(RUN_DIR, RUN_ID, OUTPUT_ROOT)

22:07:21 [INFO] Refreshing latest pointers ...
22:07:21 [INFO] Latest pointer refreshed: trends24_id_bumpchart_20260721-140704.png -> latest_light.png
22:07:21 [INFO] Latest pointer refreshed: trends24_id_bumpchart_20260721-140704_dark.png -> latest_dark.png
22:07:21 [INFO] Latest pointer refreshed: trends24_id_bumpchart_20260721-140704_interactive.html -> latest_interactive.html


In [27]:
print("=" * 72)
print(f"Run ID         : {RUN_ID}")
print(f"Captured at    : {RUN_TS.isoformat()}")
print(f"Repo root      : {REPO_ROOT}")
print(f"Total rows     : {total_rows}")
print(f"Unique topics  : {unique_topics}")
print(f"Null joins     : {null_join_count} rows missing detail-stats fields")
print(f"Hour-columns   : {len(timeline_columns)}")
print(f"Combined rows  : {len(df_combined)} (across all runs, after dedupe)")
print(f"Combined CSV   : {COMBINED_CSV_PATH}")
print(f"CSV            : {CSV_PATH}")
print(f"Light PNG      : {PNG_PATH}")
print(f"Dark PNG       : {PNG_DARK_PATH}")
print(f"Interactive    : {HTML_PATH}")
print(f"Latest light   : {OUTPUT_ROOT / 'latest_light.png'}")
print(f"Latest dark    : {OUTPUT_ROOT / 'latest_dark.png'}")
print(f"Latest HTML    : {OUTPUT_ROOT / 'latest_interactive.html'}")
print("=" * 72)

Run ID         : 20260721-140704
Captured at    : 2026-07-21T14:07:04.852778+00:00
Repo root      : X:\githubrepo\kestrel-trending
Total rows     : 1197
Unique topics  : 147
Null joins     : 1197 rows missing detail-stats fields
Hour-columns   : 24
Combined rows  : 2575 (across all runs, after dedupe)
Combined CSV   : X:\githubrepo\kestrel-trending\dataset\trends\combined\trends24_id_combined.csv
CSV            : X:\githubrepo\kestrel-trending\dataset\trends\run_20260721-140704\trends24_id_raw_20260721-140704.csv
Light PNG      : X:\githubrepo\kestrel-trending\dataset\trends\run_20260721-140704\trends24_id_bumpchart_20260721-140704.png
Dark PNG       : X:\githubrepo\kestrel-trending\dataset\trends\run_20260721-140704\trends24_id_bumpchart_20260721-140704_dark.png
Interactive    : X:\githubrepo\kestrel-trending\dataset\trends\run_20260721-140704\trends24_id_bumpchart_20260721-140704_interactive.html
Latest light   : X:\githubrepo\kestrel-trending\dataset\trends\latest_light.png
Latest d